In [ ]:
pip install --pre -U langchain langchain-openai

In [4]:

import os
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [5]:
pdf_url = "https://arxiv.org/pdf/2501.04040.pdf"
loader = PyPDFLoader(pdf_url)

documents = loader.load()

In [4]:
documents[0]
print(f"\n✅ loaded {len(documents)} pages from pdf")


✅ loaded 174 pages from pdf


In [6]:
sample_doc = documents[0]
print(f"\nSample Document Structure:")
print(f"- Content length : {len(sample_doc.page_content)} character's")
print(f"- Metadata: {sample_doc.metadata}")
print(f"- Content Preview: {sample_doc.page_content[:200]}...")


Sample Document Structure:
- Content length : 2140 character's
- Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-11T01:48:37+00:00', 'author': '', 'keywords': '', 'moddate': '2025-02-11T01:48:37+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/2501.04040.pdf', 'total_pages': 174, 'page': 0, 'page_label': '1'}
- Content Preview: A Survey on Large Language Models with some Insights
on their Capabilities and Limitations
Andrea Matarazzo
Expedia Group
Italy
a.matarazzo@gmail.com
Riccardo Torlone
Roma Tre University
Italy
riccard...


In [7]:
# Text Spliting

# Configuring Text Spliting
# Chunk size 1024 characters
# overlap 10%
# Method Recursive Character spliting

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1024,
    # 10% 
    chunk_overlap = 100, 
    length_function=len,
    add_start_index=True
)

print(f"\n Splitting documents into chunks ")
chunks = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} pages into {len(chunks)} chunks")

chunk_size = [len(chunk.page_content) for chunk in chunks]

print(f"Analysis chunks: ")
print(f"- Average chunk size: {sum(chunk_size) / len(chunk_size):.0f} characters")
print(f"-Largest chunk : {max(chunk_size)} characters")
print(f"Smallest chunk: {min(chunk_size)} characters")


 Splitting documents into chunks 
✅ Split 174 pages into 593 chunks
Analysis chunks: 
- Average chunk size: 858 characters
-Largest chunk : 1024 characters
Smallest chunk: 25 characters


In [8]:
# Embedding

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

In [10]:
len(embeddings.embed_query("Hello world"))

768

In [9]:
# Vector Stores

# Creating Chroma Vector Store
# Collection name: pdf collection 
# Storage : Local persistent directory
# Embedding function: nomic-embed-text via ollama

vector_store = Chroma(
    collection_name= "pdf_collection",
    embedding_function=embeddings,
    persist_directory="./chroma.db"
)

vector_store.add_documents(documents=chunks)

print(f"✅ Added {len(chunks)} document chunks to vector store")


✅ Added 593 document chunks to vector store


In [10]:
# Querying The vector Store
print("\n Basic similarity search.")
print("Finding documents most similar to a query using cosine similarity")

query = "What is main method available for RAG?"

results = vector_store.similarity_search(query, k = 5)

print(f"\nQuery: '{query}'")
print(f"Retrieved {len(results)} most similar chunks: ")

for i, doc in enumerate(results, 1):
    print(f"\n --Result {i} ---")
    print(f"Content: {doc.page_content[:300]}...")
    print(f"Source: Page {doc.metadata.get('page', 'unknown')}")


 Basic similarity search.
Finding documents most similar to a query using cosine similarity

Query: 'What is main method available for RAG?'
Retrieved 5 most similar chunks: 

 --Result 1 ---
Content: sions of Wikipedia are widely used in most LLMs (e.g., GPT-3 [88], and LLaMA [330]).
Wikipedia is available in multiple languages and can be used in multilingual settings.
• Code: Two major sources are GitHub, for open-source licensed code, and StackOver-
flow, for code-related question-answering pl...
Source: Page 48

 --Result 2 ---
Content: sions of Wikipedia are widely used in most LLMs (e.g., GPT-3 [88], and LLaMA [330]).
Wikipedia is available in multiple languages and can be used in multilingual settings.
• Code: Two major sources are GitHub, for open-source licensed code, and StackOver-
flow, for code-related question-answering pl...
Source: Page 48

 --Result 3 ---
Content: method termed “Evol-Instruct”, which significantly enhances the instruction-following
capabilities and ove

In [12]:
print(f"Similarity Search with scores")
print(f"Same search but with similarity score to see confidence levels...")

results_with_scores = vector_store.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n --- Results {i} (similarity score: {score:.4f}) ---")
    print(f"Content: {doc.page_content[:200]} ....")
    print(f"Source: Page {doc.metadata.get('page', 'unknown')}")

Similarity Search with scores
Same search but with similarity score to see confidence levels...

 --- Results 1 (similarity score: 0.7216) ---
Content: sions of Wikipedia are widely used in most LLMs (e.g., GPT-3 [88], and LLaMA [330]).
Wikipedia is available in multiple languages and can be used in multilingual settings.
• Code: Two major sources ar ....
Source: Page 48

 --- Results 2 (similarity score: 0.7216) ---
Content: sions of Wikipedia are widely used in most LLMs (e.g., GPT-3 [88], and LLaMA [330]).
Wikipedia is available in multiple languages and can be used in multilingual settings.
• Code: Two major sources ar ....
Source: Page 48

 --- Results 3 (similarity score: 0.7286) ---
Content: method termed “Evol-Instruct”, which significantly enhances the instruction-following
capabilities and overall performance of large language models (LLMs). It is a systematic
approach for automaticall ....
Source: Page 58

 --- Results 4 (similarity score: 0.7286) ---
Content: method termed 

In [13]:
# meta filtering
# Using meta filters to search specific parts of the document..
print("\n Meta Filtering")
print("Using metadata filters to search specific parts of the document. ")

# First let's see what meta data is available 
print(f"\nAvailable meta data for our chunk")

if chunks: 
    sample_metadata = chunks[0].metadata
    print(f"Sample metadata: {sample_metadata}")
    
    # Get unique page numbers for filtering examples
    page_number = set()
    
    for chunk in chunks[:10]:
        if 'page' in chunk.metadata:
            page_number.add(chunk.metadata['page'])
    
    print(f"Available page numbers (sample): {sorted(list(page_number))[:5]}...")


 Meta Filtering
Using metadata filters to search specific parts of the document. 

Available meta data for our chunk
Sample metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-11T01:48:37+00:00', 'author': '', 'keywords': '', 'moddate': '2025-02-11T01:48:37+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/2501.04040.pdf', 'total_pages': 174, 'page': 0, 'page_label': '1', 'start_index': 0}
Available page numbers (sample): [0, 1, 2]...


In [14]:
# Filter by specific page
print(f"\nFilter by specific page")

if page_number:
    target_page = sorted(list(page_number))[0]
    page_result = vector_store.similarity_search(
        "methodology approach",
        k=10,
        filter={"page": target_page}
    )
    print(f"Search only in Page {target_page}:")
    for i, doc in enumerate(page_result, 1):
        print(f" Result {i}: Page {doc.metadata.get('page')} - {doc.page_content[:150]}...")


Filter by specific page
Search only in Page 0:
 Result 1: Page 0 - architectural strategies that drive these capabilities. Emphasizing models like GPT and
LLaMA, we analyze the impact of exponential data and computati...
 Result 2: Page 0 - architectural strategies that drive these capabilities. Emphasizing models like GPT and
LLaMA, we analyze the impact of exponential data and computati...
 Result 3: Page 0 - frameworks that integrate external systems, allowing LLMs to handle complex, dynamic
tasks. By analyzing these factors, this paper aims to foster the ...
 Result 4: Page 0 - frameworks that integrate external systems, allowing LLMs to handle complex, dynamic
tasks. By analyzing these factors, this paper aims to foster the ...
 Result 5: Page 0 - A Survey on Large Language Models with some Insights
on their Capabilities and Limitations
Andrea Matarazzo
Expedia Group
Italy
a.matarazzo@gmail.com
...
 Result 6: Page 0 - A Survey on Large Language Models with some Insights
on their 

In [ ]:
# multiple Metadata filter

print(f"Multiple data filter!")

complex_results = vector_store.similarity_search(
    "research findings",
    k = 2,
    filter={
        "$and": [
            {"page": {"$lte" : 10}}, # lte => less then or equal to
            {"source": {"$ne": ""}} # ne => not equal
        ]
    }
)

print("Using Complex filter (page>=0) and has source")
for i, doc in enumerate(complex_results, 1):
    print(f" Result {i}: Page {doc.metadata.get('page')} - {doc.page_content[:150]}...")


Multiple data filter!
Using Complex filter (page>=0) and has source
 Result 1: Page 4 - the transformative impact of LLMs across various domains, including healthcare, finance,
education, law, and scientific research.
• Section 3 focuses ...
 Result 2: Page 4 - the transformative impact of LLMs across various domains, including healthcare, finance,
education, law, and scientific research.
• Section 3 focuses ...


In [ ]:
# Retrievers
print(f"\n Create Retriever ")

similarity_retriever = vector_store.as_retriever(
    search_type ="similarity",
    search_kwargs = {"k": 4}
)